# Gridlock Traffic Detector GPU Training

Before running: Colab menu -> Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi

Upload `gridlock_colab_training_package.zip` to Google Drive root: `MyDrive/gridlock_colab_training_package.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from pathlib import Path
import shutil

drive_zip = Path('/content/drive/MyDrive/gridlock_colab_training_package.zip')
work_dir = Path('/content/gridlock')

assert drive_zip.exists(), f'Missing package: {drive_zip}'
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

print('Package found:', drive_zip)
print('Work dir:', work_dir)

In [ ]:
!unzip -q "/content/drive/MyDrive/gridlock_colab_training_package.zip" -d /content/gridlock
!find /content/gridlock -maxdepth 4 -type f -name 'dataset.yaml' -o -name '*.pt' | head -20

In [ ]:
dataset_yaml = Path('/content/gridlock/data/processed/traffic_multitask_detector_v2/dataset.yaml')
dataset_root = Path('/content/gridlock/data/processed/traffic_multitask_detector_v2')
base_model = Path('/content/gridlock/backend/yolo26s.pt')

assert dataset_yaml.exists(), dataset_yaml
assert base_model.exists(), base_model

lines = []
for line in dataset_yaml.read_text().splitlines():
    if line.startswith('path:'):
        lines.append(f'path: {dataset_root}')
    else:
        lines.append(line)
dataset_yaml.write_text('\n'.join(lines) + '\n')
print(dataset_yaml.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO(str(base_model))
results = model.train(
    data=str(dataset_yaml),
    epochs=80,
    imgsz=960,
    batch=-1,
    device=0,
    project='/content/drive/MyDrive/gridlock_runs',
    name='final_multitask_yolo26s',
    patience=20,
    cos_lr=True,
    close_mosaic=15,
    multi_scale=0.15,
    cache=True,
    plots=True,
)

If the previous cell fails with CUDA out-of-memory, run this smaller version instead.

In [ ]:
# OOM fallback only: run this cell if 960 image size fails.
# from ultralytics import YOLO
# model = YOLO(str(base_model))
# results = model.train(
#     data=str(dataset_yaml), epochs=80, imgsz=768, batch=-1, device=0,
#     project='/content/drive/MyDrive/gridlock_runs', name='final_multitask_yolo26s_768',
#     patience=20, cos_lr=True, close_mosaic=15, multi_scale=0.15, cache=True, plots=True,
# )

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path('/content/drive/MyDrive/gridlock_runs/final_multitask_yolo26s')
best = run_dir / 'weights' / 'best.pt'
if not best.exists():
    run_dir = Path('/content/drive/MyDrive/gridlock_runs/final_multitask_yolo26s_768')
    best = run_dir / 'weights' / 'best.pt'
assert best.exists(), f'Missing best checkpoint: {best}'

model = YOLO(str(best))
metrics = model.val(data=str(dataset_yaml), imgsz=960 if '768' not in str(best) else 768, device=0, plots=True)
print('best:', best)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('names:', metrics.names)

In [ ]:
export_path = Path('/content/drive/MyDrive/gridlock_traffic_yolo26s_final.pt')
shutil.copy2(best, export_path)
print('Copied final model to:', export_path)